# 00 — Setup dan Jalankan Semua

**Buka notebook ini, tekan Run All, selesai.** Tidak ada perintah terminal yang perlu
Anda jalankan sendiri.

Notebook ini mengerjakan:

1. Memasang seluruh dependency Python
2. Memeriksa GPU, dan memasang ulang PyTorch versi CUDA kalau yang terpasang versi CPU
3. Memasang kredensial Kaggle dari file `kaggle.json` yang Anda unduh
4. Menjalankan sepuluh self-check modul — sebelum mengunduh apa pun
5. Menjalankan notebook 01 sampai 06 berurutan, dengan output tersimpan di dalamnya

Model spaCy dan FastText diunduh sendiri saat pertama dibutuhkan.

## Satu hal yang harus ada lebih dulu

Jupyter itu sendiri. Kalau Anda bisa membaca notebook ini, syarat itu sudah
terpenuhi — lewat Anaconda, VS Code, atau `jupyter lab`.

## Sekali lagi soal durasi

Setelan paper penuh (100/200/200 epoch, 3 seed) di GPU kelas RTX 3050 perkiraannya
**12-18 jam**. Sel terakhir punya pilihan `CEPAT = True` untuk menguji seluruh
pipeline dalam ± 1 jam lebih dulu. **Pakai itu pada percobaan pertama.**

## 1. Dependency

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("root proyek:", ROOT)
print("python     :", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    print("\nCATATAN: proyek ini diuji pada Python 3.12. Versi lain mungkin belum "
          "punya wheel untuk torch/spacy/fasttext.")

In [ ]:
%pip install -q -r {ROOT / "requirements.txt"}
print("dependency selesai dipasang")

## 2. GPU dan PyTorch

Pemeriksaan dijalankan di **subprocess**, bukan di kernel ini. Alasannya praktis:
kalau kernel sudah meng-import torch, memasang ulang torch butuh restart kernel.
Dengan subprocess, kernel ini tidak pernah menyentuh torch sampai semuanya beres.

In [ ]:
def cek_torch():
    r = subprocess.run(
        [sys.executable, "-c",
         "import torch,json;"
         "d={'versi':torch.__version__,'cuda':torch.cuda.is_available()};"
         "d['nama']=torch.cuda.get_device_properties(0).name if d['cuda'] else None;"
         "d['vram_gb']=round(torch.cuda.get_device_properties(0).total_memory/1e9,1)"
         " if d['cuda'] else None;"
         "print(json.dumps(d))"],
        capture_output=True, text=True)
    if r.returncode != 0:
        return None
    import json
    return json.loads(r.stdout.strip().splitlines()[-1])

def ada_gpu_nvidia():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

info = cek_torch()
print("torch:", info)
print("nvidia-smi terdeteksi:", ada_gpu_nvidia())

In [ ]:
# Kalau ada GPU NVIDIA tapi torch-nya versi CPU, pasang ulang dari indeks CUDA.
# Tanpa ini semuanya tetap jalan, hanya puluhan kali lebih lambat.
if info and not info["cuda"] and ada_gpu_nvidia():
    print("GPU NVIDIA ada tapi torch tidak melihat CUDA. Memasang ulang...\n")
    %pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124
    info = cek_torch()
    print("\nsetelah dipasang ulang:", info)
    if info and info["cuda"]:
        print("\nBERHASIL. Lanjutkan ke sel berikutnya.")
    else:
        print("\nMasih belum terdeteksi. Periksa versi driver dengan nvidia-smi, "
              "lalu ganti cu124 dengan versi yang sesuai (cu121, cu126, ...).")
elif info and info["cuda"]:
    print(f"CUDA siap: {info['nama']}, {info['vram_gb']} GB VRAM")
    if info["vram_gb"] < 5:
        print("VRAM di bawah 5 GB — micro-batch akan otomatis diset 8 dengan "
              "akumulasi gradien, jadi batch efektif tetap 40 seperti paper.")
else:
    print("Tidak ada GPU NVIDIA. Training akan berjalan di CPU dan SANGAT lambat.")
    print("Sangat disarankan memakai CEPAT = True di sel terakhir.")

## 3. Kredensial Kaggle

Dataset citra diunduh dari Kaggle, yang butuh API token.

**Sekali saja:** buka kaggle.com → klik foto profil → Settings → bagian API →
*Create New Token*. Browser mengunduh `kaggle.json` yang isinya seperti ini:

```json
{"username":"namaanda","key":"0123456789abcdef0123456789abcdef"}
```

Sel di bawah pertama-tama mencari file itu di folder Downloads dan di folder proyek.
Kalau tidak ada, ia **meminta Anda mengetikkannya langsung di sini** — tidak perlu
buka file manager.

> **Kenapa pakai `getpass` dan bukan `input()`.** Repo ini publik di GitHub, dan
> output notebook ikut ter-commit. `input()` menampilkan apa yang Anda ketik,
> sehingga token akan tersimpan di dalam file `.ipynb` lalu ter-push ke publik.
> `getpass` menyembunyikannya, jadi tidak ada jejak di output. Token juga ditulis ke
> `~/.kaggle/kaggle.json` di luar folder proyek, bukan ke dalam repo.

In [ ]:
import getpass, json, os, shutil, stat

tujuan = Path.home() / ".kaggle" / "kaggle.json"

def _kunci(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        path.chmod(stat.S_IRUSR | stat.S_IWUSR)   # 600; diabaikan di Windows
    except OSError:
        pass

if tujuan.exists() or os.environ.get("KAGGLE_KEY"):
    print(f"kredensial Kaggle sudah ada, tidak perlu diisi lagi ({tujuan})")
else:
    # Jalur termudah: file hasil unduhan masih di tempatnya.
    kandidat = [p for d in (Path.home() / "Downloads", Path.home() / "Unduhan", ROOT)
                for p in [d / "kaggle.json"] if p.exists()]
    if kandidat:
        asal = kandidat[0]
        tujuan.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(asal, tujuan)
        _kunci(tujuan)
        print(f"kaggle.json ditemukan di {asal} dan dipasang ke {tujuan}")
        if asal == ROOT / "kaggle.json":
            asal.unlink()   # jangan tinggalkan token di dalam repo
            print("salinan di folder proyek dihapus supaya tidak ikut ter-commit")
    else:
        print("kaggle.json tidak ditemukan. Ketikkan manual di bawah.")
        print("Nilainya ada di file kaggle.json hasil unduhan, atau di "
              "kaggle.com -> Settings -> API.\n")
        username = input("Kaggle username: ").strip()
        # getpass: tidak ditampilkan, jadi tidak tersimpan di output notebook
        key = getpass.getpass("Kaggle key (tidak akan terlihat saat diketik): ").strip()

        if not username or not key:
            print("\nBatal — username atau key kosong. Jalankan ulang sel ini.")
        else:
            tujuan.parent.mkdir(parents=True, exist_ok=True)
            tujuan.write_text(json.dumps({"username": username, "key": key}),
                              encoding="utf-8")
            _kunci(tujuan)
            # sengaja tidak mencetak nilainya, hanya panjangnya sebagai konfirmasi
            print(f"\nkredensial ditulis ke {tujuan} "
                  f"(username {username}, key {len(key)} karakter)")
            del key

In [ ]:
# Uji kredensialnya sekarang, bukan nanti setelah 20 menit di notebook 01.
r = subprocess.run(
    [sys.executable, "-c",
     "import kagglehub;"
     "from kagglehub.auth import whoami;"
     "print('kaggle ok:', whoami().get('username'))"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip().splitlines()[-1])
if r.returncode != 0:
    print("\nKredensial belum bisa dipakai. Periksa username/key, atau buat token "
          "baru di kaggle.com -> Settings -> API -> Create New Token.")

## 4. Self-check

Sepuluh modul di `src/` punya pemeriksaan yang berjalan **tanpa dataset**: bentuk
tensor, aliran gradien, logika split, akumulasi gradien, dan seterusnya.

Dijalankan sekarang, sebelum mengunduh 3,3 GB — kalau ada yang rusak, tidak ada
gunanya mengunduh apa pun.

In [ ]:
sys.path.insert(0, str(ROOT))
import run_all

galat = run_all.periksa_prasyarat()
lolos = run_all.jalankan_self_check()

if galat:
    print("\nPrasyarat belum lengkap:")
    for g in galat:
        print(f"  - {g}")
if not lolos:
    print("\nSelf-check gagal — perbaiki dulu sebelum lanjut.")
if lolos and not galat:
    print("\nSemuanya siap.")

## 5. Jalankan seluruh pipeline

Ubah `CEPAT` di bawah:

- `CEPAT = True` — 15/20/20 epoch, 1 seed, ± 1 jam. **Pakai ini dulu.** Angkanya
  tidak sah untuk laporan, tapi ia membuktikan seluruh pipeline jalan. Deviasinya
  dicatat otomatis sehingga tidak bisa tersalahartikan sebagai hasil penuh.
- `CEPAT = False` — setelan paper penuh, 12-18 jam di RTX 3050.

Kalau ada notebook yang gagal, notebook itu menyimpan traceback-nya di sel yang
gagal. Perbaiki, lalu ubah `DARI` ke nomor notebook itu dan jalankan sel ini lagi —
yang sudah selesai tidak diulang.

In [ ]:
CEPAT = True      # ubah ke False untuk setelan paper penuh
DARI = "01"       # lanjutkan dari notebook tertentu setelah kegagalan

if CEPAT:
    os.environ["APDM_CEPAT"] = "1"
else:
    os.environ.pop("APDM_CEPAT", None)

import importlib
from src import config as _cfg
importlib.reload(_cfg)          # supaya perubahan APDM_CEPAT terbaca
print("epoch:", _cfg.CONFIG["epochs"], "| seed:", _cfg.CONFIG["seeds"])
print("micro-batch:", _cfg.CONFIG["micro_batch_size"] or _cfg.CONFIG["batch_size"])
print("\nDeviasi runtime yang akan berlaku:")
for d in _cfg.deviasi_runtime():
    print(f"  - {d}")

In [ ]:
import time

mulai = [i for i, (n, _, _) in enumerate(run_all.NOTEBOOKS) if n == DARI][0]
t0 = time.perf_counter()
gagal = None

for _, nama, deskripsi in run_all.NOTEBOOKS[mulai:]:
    if not run_all.jalankan_notebook(nama, deskripsi):
        gagal = nama
        break

jam = (time.perf_counter() - t0) / 3600
if gagal:
    print(f"\nBerhenti di {gagal} setelah {jam:.2f} jam.")
    print(f"Buka notebooks/{gagal}.ipynb, perbaiki, lalu set DARI = \"{gagal[:2]}\" "
          f"dan jalankan ulang dua sel terakhir.")
else:
    print(f"\nSELESAI dalam {jam:.2f} jam.")
    print("\nAngkanya ada di sel ringkasan terakhir tiap notebook, plus di reports/.")
    print("Langkah terakhir manual: salin ke catatan di vault/30-eksperimen/ dan "
          "vault/50-hasil/, lalu ubah status belum-dijalankan menjadi selesai.")
    print("\nItu satu-satunya yang tidak diotomatiskan, dan memang sengaja: kalau "
          "angka disalin otomatis, aturan 'jangan pernah mengarang angka' jadi tidak "
          "bisa diverifikasi lagi.")

## Kalau ada yang tidak berjalan

Rinciannya di [RUNBOOK.md](../RUNBOOK.md). Yang paling sering:

| Gejala | Penyebab |
|---|---|
| `OutOfMemoryError` di notebook 04/05 | restart kernel, lalu set `CONFIG["micro_batch_size"] = 4` di sel pertama notebook itu |
| Notebook 01 gagal di `assert rec["n_kelas_cocok"] == 10` | nama folder kelas di arsip belum ada di `CLASS_ALIASES`; tambahkan di `src/data.py`, jangan diakali di notebook |
| Notebook 02 kehabisan RAM | FastText butuh ± 15 GB; tutup aplikasi lain. Jangan pakai `reduce_model` — ia tetap memuat model penuh lebih dulu |
| `DataLoader` menggantung di Windows | sudah ditangani (`num_workers=0` otomatis di Windows); kalau tetap terjadi, pakai WSL2 |